# 05 · Merge & Calendar Enrichment

**Inputs :**
- `data/processed/panel_daily.parquet`  
- `data/processed/weather_daily.parquet` 

**Output :** `data/processed/analysis_panel.parquet`

Steps:
0. Imports & load data
1. Merge cycling + weather
2. Belgian public holidays
3. Flemish school holidays
4. KU Leuven academic calendar
5. Combined calendar flags
6. Lagged weather variables
7. Quality check & date filter
8. Save final analysis panel
9.  Final summary

## 0. Imports & Load Data

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "holidays"])

from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import holidays

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

ROOT      = Path().resolve().parent
PROCESSED = ROOT / "data" / "processed"
FIGS      = ROOT / "outputs" / "figures"

cycling = pd.read_parquet(PROCESSED / "panel_daily.parquet")
weather = pd.read_parquet(PROCESSED / "weather_daily.parquet")

# Guard against date32 / object dtype from parquet readers
cycling["date"] = pd.to_datetime(cycling["date"])
weather["date"] = pd.to_datetime(weather["date"])

print(f"cycling : {cycling.shape}")
print(f"weather : {weather.shape}")
print(f"cycling date range : {cycling['date'].min().date()} -> {cycling['date'].max().date()}")
print(f"weather date range : {weather['date'].min().date()} -> {weather['date'].max().date()}")

cycling : (164400, 19)
weather : (164400, 19)
cycling date range : 2023-01-01 -> 2025-12-31
weather date range : 2023-01-01 -> 2025-12-31


## 1. Merge Cycling + Weather

In [2]:
n_before = len(cycling)

merged = cycling.merge(
    weather,
    on=["site ID", "date"],
    how="left",
    validate="1:1",
)

n_nan_temp = merged["temp_avg"].isna().sum()
n_dup      = merged.duplicated(["site ID", "date"]).sum()

print(f"cycling rows        : {n_before:,}")
print(f"merged rows         : {len(merged):,}  (expected: {n_before:,})")
print(f"NaN in temp_avg     : {n_nan_temp:,}  ({n_nan_temp / len(merged) * 100:.1f}%)")
print(f"Duplicate (site,date): {n_dup}  (should be 0)")

cycling rows        : 164,400
merged rows         : 164,400  (expected: 164,400)
NaN in temp_avg     : 0  (0.0%)
Duplicate (site,date): 0  (should be 0)


## 2. Add Belgian Public Holidays

In [3]:
be_holidays = holidays.Belgium(years=[2023, 2024, 2025])

holiday_dates = (
    pd.DataFrame([
        {"date": pd.Timestamp(d), "public_holiday_name": name}
        for d, name in be_holidays.items()
    ])
    .sort_values("date")
    .reset_index(drop=True)
)

n_before = len(merged)
merged = merged.merge(holiday_dates, on="date", how="left")
merged["is_public_holiday"] = merged["public_holiday_name"].notna()
assert len(merged) == n_before, "Public holiday merge duplicated rows!"

print(f"Belgian public holidays 2023-2025 ({len(holiday_dates)} total):")
for _, row in holiday_dates.iterrows():
    print(f"  {str(row['date'].date()):<12}  {row['public_holiday_name']}")

Belgian public holidays 2023-2025 (36 total):
  2023-01-01    Nieuwjaar
  2023-04-09    Pasen
  2023-04-10    Paasmaandag
  2023-05-01    Dag van de Arbeid
  2023-05-18    O. L. H. Hemelvaart
  2023-05-28    Pinksteren
  2023-05-29    Pinkstermaandag
  2023-07-21    Nationale feestdag
  2023-08-15    O. L. V. Hemelvaart
  2023-11-01    Allerheiligen
  2023-11-11    Wapenstilstand
  2023-12-25    Kerstmis
  2024-01-01    Nieuwjaar
  2024-03-31    Pasen
  2024-04-01    Paasmaandag
  2024-05-01    Dag van de Arbeid
  2024-05-09    O. L. H. Hemelvaart
  2024-05-19    Pinksteren
  2024-05-20    Pinkstermaandag
  2024-07-21    Nationale feestdag
  2024-08-15    O. L. V. Hemelvaart
  2024-11-01    Allerheiligen
  2024-11-11    Wapenstilstand
  2024-12-25    Kerstmis
  2025-01-01    Nieuwjaar
  2025-04-20    Pasen
  2025-04-21    Paasmaandag
  2025-05-01    Dag van de Arbeid
  2025-05-29    O. L. H. Hemelvaart
  2025-06-08    Pinksteren
  2025-06-09    Pinkstermaandag
  2025-07-21    Nationale

## 3. Add Flemish School Holidays

In [4]:
# Official dates from [onderwijs.vlaanderen.be](https://onderwijs.vlaanderen.be/nl/schoolvakanties).
school_holidays = [
    # School year 2022-2023 (2023 portion within our window)
    ("2023-01-01", "2023-01-08",  "christmas_2022"),   # Kerstvakantie (end)
    ("2023-02-20", "2023-02-26",  "spring_2023"),       # Krokusvakantie
    ("2023-04-03", "2023-04-16",  "easter_2023"),       # Paasvakantie
    ("2023-07-01", "2023-08-31",  "summer_2023"),       # Zomervakantie
    # School year 2023-2024
    ("2023-10-30", "2023-11-05",  "autumn_2023"),       # Herfstvakantie
    ("2023-12-25", "2023-12-31",  "christmas_2023"),    # Kerstvakantie (start)
    ("2024-01-01", "2024-01-07",  "christmas_2023"),    # Kerstvakantie (end)
    ("2024-02-12", "2024-02-18",  "spring_2024"),       # Krokusvakantie
    ("2024-04-01", "2024-04-14",  "easter_2024"),       # Paasvakantie
    ("2024-07-01", "2024-08-31",  "summer_2024"),       # Zomervakantie
    ("2024-10-28", "2024-11-03",  "autumn_2024"),       # Herfstvakantie
    # School year 2024-2025
    ("2024-12-23", "2025-01-05",  "christmas_2024"),
    ("2025-03-03", "2025-03-09",  "spring_2025"),
    ("2025-04-07", "2025-04-21",  "easter_2025"),
    ("2025-07-01", "2025-08-31",  "summer_2025"),
    ("2025-10-27", "2025-11-02",  "autumn_2025"),
    ("2025-12-22", "2025-12-31",  "christmas_2025"),
]

school_rows = []
for start, end, name in school_holidays:
    for d in pd.date_range(start, end, freq="D"):
        school_rows.append({"date": d, "school_holiday_name": name})

school_df = (
    pd.DataFrame(school_rows)
    .drop_duplicates("date", keep="first")  # christmas_2023 spans Dec+Jan, avoid overlap
    .reset_index(drop=True)
)

n_before = len(merged)
merged = merged.merge(school_df, on="date", how="left")
merged["is_school_holiday"] = merged["school_holiday_name"].notna()
assert len(merged) == n_before, "School holiday merge duplicated rows!"

print("School holiday days per period:")
counts = school_df.groupby("school_holiday_name").size().sort_index()
for name, n in counts.items():
    print(f"  {name:<22} : {n:>3} days")
print(f"  {'TOTAL':<22} : {len(school_df):>3} calendar days")

School holiday days per period:
  autumn_2023            :   7 days
  autumn_2024            :   7 days
  autumn_2025            :   7 days
  christmas_2022         :   8 days
  christmas_2023         :  14 days
  christmas_2024         :  14 days
  christmas_2025         :  10 days
  easter_2023            :  14 days
  easter_2024            :  14 days
  easter_2025            :  15 days
  spring_2023            :   7 days
  spring_2024            :   7 days
  spring_2025            :   7 days
  summer_2023            :  62 days
  summer_2024            :  62 days
  summer_2025            :  62 days
  TOTAL                  : 317 calendar days


## 4. Add KU Leuven Academic Calendar

In [5]:
# Official dates from [kuleuven.be](https://www.kuleuven.be/onderwijs/academische-kalender).
# Note: 2025-02-01 appears in both the `exam` and `recess` windows — the overlap is resolved by
# keeping `exam` (first occurrence). Some calendar days have no period entry due to gaps in the
# published schedule (e.g. 2024-06-30, 2024-09-08, 2025-06-09, 2025-06-29).

ku_leuven_periods = [
    # Academic year 2022-2023 (2023 portion)
    ("2023-01-01", "2023-01-08",  "recess"),       # Christmas holiday (end)
    ("2023-01-09", "2023-01-12",  "exam_prep"),    # Study/block period
    ("2023-01-13", "2023-02-04",  "exam"),          # First examination period
    ("2023-02-05", "2023-02-12",  "recess"),        # No classes week
    ("2023-02-13", "2023-03-31",  "teaching"),      # Second semester
    ("2023-04-01", "2023-04-16",  "recess"),        # Easter holiday
    ("2023-04-17", "2023-05-26",  "teaching"),      # Teaching continues
    ("2023-05-27", "2023-06-11",  "exam_prep"),     # Study/block period
    ("2023-06-12", "2023-07-01",  "exam"),          # Second examination period
    ("2023-07-02", "2023-08-20",  "recess"),        # Summer recess
    ("2023-08-21", "2023-09-09",  "exam"),          # Third examination period
    ("2023-09-10", "2023-09-24",  "recess"),        # Board meetings + orientation
    # Academic year 2023-2024 (2023 portion)
    ("2023-09-25", "2023-12-22",  "teaching"),      # First semester
    ("2023-12-23", "2023-12-31",  "recess"),        # Christmas holiday (start)
    # Academic year 2023-2024 (2024 portion)
    ("2024-01-01", "2024-01-07",  "recess"),
    ("2024-01-08", "2024-01-14",  "exam_prep"),
    ("2024-01-15", "2024-02-03",  "exam"),          # First exam
    ("2024-02-04", "2024-02-11",  "recess"),
    ("2024-02-12", "2024-03-29",  "teaching"),      # Second semester
    ("2024-03-30", "2024-04-14",  "recess"),        # Easter holiday
    ("2024-04-15", "2024-05-24",  "teaching"),
    ("2024-05-25", "2024-06-09",  "exam_prep"),
    ("2024-06-10", "2024-06-29",  "exam"),          # Second exam
    ("2024-06-30", "2024-08-18",  "recess"),
    ("2024-08-19", "2024-09-07",  "exam"),          # Third exam
    ("2024-09-08", "2024-09-22",  "recess"),
    # Academic year 2024-2025
    ("2024-09-23", "2024-12-20",  "teaching"),      # First semester
    ("2024-12-21", "2025-01-05",  "recess"),        # Christmas holiday
    ("2025-01-06", "2025-01-12",  "exam_prep"),
    ("2025-01-13", "2025-02-01",  "exam"),          # First exam
    ("2025-02-01", "2025-02-09",  "recess"),        # 2025-02-01 overlaps with exam above
    ("2025-02-10", "2025-04-04",  "teaching"),      # Second semester
    ("2025-04-05", "2025-04-21",  "recess"),
    ("2025-04-22", "2025-05-23",  "teaching"),
    ("2025-05-24", "2025-06-09",  "exam_prep"),
    ("2025-06-10", "2025-06-28",  "exam"),
    ("2025-06-29", "2025-08-17",  "recess"),
    ("2025-08-18", "2025-09-06",  "exam"),
    ("2025-09-07", "2025-12-31",  "teaching"),
]

ku_rows = []
for start, end, period in ku_leuven_periods:
    for d in pd.date_range(start, end, freq="D"):
        ku_rows.append({"date": d, "ku_leuven_period": period})

# Drop duplicate dates — keep first occurrence (exam takes precedence over recess)
ku_df = (
    pd.DataFrame(ku_rows)
    .drop_duplicates("date", keep="first")
    .reset_index(drop=True)
)

n_before = len(merged)
merged = merged.merge(ku_df, on="date", how="left")
assert len(merged) == n_before, "KU Leuven merge duplicated rows!"

merged["ku_is_teaching"]  = merged["ku_leuven_period"] == "teaching"
merged["ku_is_exam"]      = merged["ku_leuven_period"] == "exam"
merged["ku_is_exam_prep"] = merged["ku_leuven_period"] == "exam_prep"
merged["ku_is_recess"]    = merged["ku_leuven_period"] == "recess"

print("KU Leuven period coverage (calendar days):")
print(ku_df["ku_leuven_period"].value_counts().sort_index())
n_total_days = len(pd.date_range("2023-01-01", "2025-12-31"))
n_gap = n_total_days - len(ku_df)
if n_gap > 0:
    gap_dates = pd.date_range("2023-01-01", "2025-12-31").difference(ku_df["date"])
    print(f"\nNote: {n_gap} date(s) have no KU Leuven period (schedule gaps):")
    print("  " + ", ".join(str(d.date()) for d in gap_dates))

KU Leuven period coverage (calendar days):
ku_leuven_period
exam         182
exam_prep     67
recess       293
teaching     554
Name: count, dtype: int64


## 5. Add Combined Calendar Flags

In [6]:
merged["is_any_holiday"] = (
    merged["is_public_holiday"] | merged["is_school_holiday"]
)

merged["is_regular_day"] = (
    ~merged["is_weekend"] &
    ~merged["is_public_holiday"] &
    ~merged["is_school_holiday"]
)

# Vectorised classify_day: public_holiday > weekend > school_holiday > regular_weekday
merged["day_type"] = np.select(
    condlist=[
        merged["is_public_holiday"],
        merged["is_weekend"],
        merged["is_school_holiday"],
    ],
    choicelist=["public_holiday", "weekend", "school_holiday"],
    default="regular_weekday",
)

print("day_type counts (site-days):")
print(merged["day_type"].value_counts())
print("\nday_type counts (unique calendar days):")
print(
    merged[["date", "day_type"]]
    .drop_duplicates("date")["day_type"]
    .value_counts()
)

day_type counts (site-days):
day_type
regular_weekday    81750
weekend            45450
school_holiday     31800
public_holiday      5400
Name: count, dtype: int64

day_type counts (unique calendar days):
day_type
regular_weekday    545
weekend            303
school_holiday     212
public_holiday      36
Name: count, dtype: int64


## 6. Add Lagged Weather Variables

In [7]:
# Sort by `(site ID, date)` first so shifts stay within each station's time series.
merged = merged.sort_values(["site ID", "date"]).reset_index(drop=True)

for lag in [1, 2]:
    merged[f"temp_avg_lag{lag}"]     = merged.groupby("site ID")["temp_avg"].shift(lag)
    merged[f"precip_total_lag{lag}"] = merged.groupby("site ID")["precip_total"].shift(lag)
    merged[f"wind_avg_lag{lag}"]     = merged.groupby("site ID")["wind_avg"].shift(lag)

lag_cols = [c for c in merged.columns if "lag" in c]
print("Lag columns:", lag_cols)

first_site = merged["site ID"].iloc[0]
print(f"\nFirst 3 days of site ID {first_site} (lag1/lag2 NaN for day 1 and 2):")
display(
    merged[merged["site ID"] == first_site]
    [["site ID", "date",
      "temp_avg", "temp_avg_lag1", "temp_avg_lag2",
      "precip_total", "precip_total_lag1", "precip_total_lag2"]]
    .head(3)
)

Lag columns: ['temp_avg_lag1', 'precip_total_lag1', 'wind_avg_lag1', 'temp_avg_lag2', 'precip_total_lag2', 'wind_avg_lag2']

First 3 days of site ID 1 (lag1/lag2 NaN for day 1 and 2):


,site ID,date,temp_avg,temp_avg_lag1,temp_avg_lag2,precip_total,precip_total_lag1,precip_total_lag2
0,1,2023-01-01,13.075000,NaN,NaN,4.4,NaN,NaN
1,1,2023-01-02,9.916667,13.075000,NaN,4.6,4.4,NaN
2,1,2023-01-03,6.866667,9.916667,13.075,0.6,4.6,4.4


## 7. Final Quality Check

In [8]:
# Date range filter
n_before = len(merged)
merged = merged[
    (merged["date"] >= "2023-01-01") &
    (merged["date"] <= "2025-12-31")
].reset_index(drop=True)

dropped = n_before - len(merged)

if dropped:
    print(f"Dropped {dropped:,} rows outside 2023-2025.")
else:
    print(f"All {len(merged):,} rows are within 2023-2025. OK")

print(f"Date range : {merged['date'].min().date()} -> {merged['date'].max().date()}")

print(f"\nAll columns ({len(merged.columns)}):")
for i, c in enumerate(merged.columns):
    print(f"  [{i:02d}] {c}")

print("\nDescriptive statistics:")
desc_cols = ["aantal", "temp_avg", "precip_total", "wind_avg",
             "humidity_avg", "sunshine_min", "peak_ratio"]
display(merged[desc_cols].describe(percentiles=[.25, .5, .75]).round(2))

# Weather coverage per site
no_wx = (
    merged.groupby("site ID")["temp_avg"]
    .apply(lambda s: s.notna().sum())
)
zero_wx = no_wx[no_wx == 0]
if len(zero_wx):
    print(f"\nWARNING: {len(zero_wx)} site(s) have zero weather coverage: {zero_wx.index.tolist()}")
else:
    print(f"\nAll {merged['site ID'].nunique()} sites have weather coverage. OK")

All 164,400 rows are within 2023-2025. OK
Date range : 2023-01-01 -> 2025-12-31

All columns (54):
  [00] site ID
  [01] date
  [02] aantal
  [03] morning_peak_count
  [04] evening_peak_count
  [05] peak_ratio
  [06] year
  [07] month
  [08] dayofweek
  [09] is_weekend
  [10] season
  [11] low_coverage
  [12] long
  [13] lat
  [14] naam
  [15] gemeente
  [16] wegnr
  [17] district
  [18] datum_van
  [19] temp_avg
  [20] temp_max
  [21] temp_min
  [22] precip_total
  [23] wind_avg
  [24] wind_max
  [25] humidity_avg
  [26] sunshine_min
  [27] rain_hours
  [28] precip_morning
  [29] wind_morning
  [30] precip_evening
  [31] wind_evening
  [32] is_rainy_day
  [33] is_cold_day
  [34] is_windy_day
  [35] is_bad_weather
  [36] public_holiday_name
  [37] is_public_holiday
  [38] school_holiday_name
  [39] is_school_holiday
  [40] ku_leuven_period
  [41] ku_is_teaching
  [42] ku_is_exam
  [43] ku_is_exam_prep
  [44] ku_is_recess
  [45] is_any_holiday
  [46] is_regular_day
  [47] day_type
  [48

,aantal,temp_avg,precip_total,wind_avg,humidity_avg,sunshine_min,peak_ratio
count,150126.00,164400.00,164400.00,164400.00,164400.00,164400.00,148281.00
mean,520.02,11.80,2.50,13.71,78.38,451.85,0.42
std,728.01,6.17,4.27,6.06,10.31,293.08,0.13
min,0.00,-4.58,0.00,1.49,34.53,0.00,0.00
25%,149.00,7.48,0.00,9.12,72.25,194.18,0.33
50%,318.00,11.71,0.50,12.69,79.54,469.97,0.41
75%,598.00,16.78,3.30,17.24,86.32,704.65,0.51
max,37188.00,29.74,66.10,45.29,99.23,960.00,1.00



All 150 sites have weather coverage. OK


## 8. Save Final Analysis Dataset

In [9]:
out = PROCESSED / "analysis_panel.parquet"
merged.to_parquet(out, index=False)

print(f"Saved : {out}")
print(f"Shape : {merged.shape}")
print("\nPreview (5 rows):")
display(
    merged[["site ID", "naam", "date", "aantal",
            "temp_avg", "precip_total",
            "day_type", "is_school_holiday",
            "ku_leuven_period", "peak_ratio"]]
    .head(5)
)

Saved : /Users/queena/Documents/KU Leuven/Study material/Semester 1.2/Modern Analysis/Project/mda-cycling-weather-group6/data/processed/analysis_panel.parquet
Shape : (164400, 54)

Preview (5 rows):


,site ID,naam,date,aantal,temp_avg,precip_total,day_type,is_school_holiday,ku_leuven_period,peak_ratio
0,1,Machelen,2023-01-01,124.0,13.075000,4.4,public_holiday,True,recess,0.161290
1,1,Machelen,2023-01-02,166.0,9.916667,4.6,school_holiday,True,recess,0.379518
2,1,Machelen,2023-01-03,354.0,6.866667,0.6,school_holiday,True,recess,0.432203
3,1,Machelen,2023-01-04,213.0,11.122917,3.4,school_holiday,True,recess,0.488263
4,1,Machelen,2023-01-05,286.0,10.295834,0.7,school_holiday,True,recess,0.500000


## 9. Final Summary

In [10]:
dates_uniq  = merged[["date", "day_type", "ku_leuven_period"]].drop_duplicates("date")
total_days  = len(dates_uniq)
dt_counts   = dates_uniq["day_type"].value_counts()
ku_counts   = dates_uniq["ku_leuven_period"].value_counts()

n_well  = merged[~merged["low_coverage"]]["site ID"].nunique()
n_low   = merged[ merged["low_coverage"]]["site ID"].nunique()
wx_cov  = merged["temp_avg"].notna().mean() * 100

print("=" * 52)
print("FINAL ANALYSIS PANEL SUMMARY")
print("=" * 52)
print(f"Shape           : {merged.shape[0]:,} rows x {merged.shape[1]} columns")
print(f"Sites           : {merged['site ID'].nunique()} "
      f"(well-covered: {n_well}, low_coverage: {n_low})")
print(f"Date range      : {merged['date'].min().date()} -> {merged['date'].max().date()}")
print()
print("Day type breakdown:")
for dt in ["regular_weekday", "weekend", "school_holiday", "public_holiday"]:
    n = dt_counts.get(dt, 0)
    print(f"  {dt:<20} : {n:>4} days ({n / total_days * 100:.1f}%)")
print()
print("KU Leuven period breakdown (calendar days):")
for p in ["teaching", "exam", "exam_prep", "recess"]:
    n = ku_counts.get(p, 0)
    print(f"  {p:<12} : {n} days")
print()
print(f"Weather coverage : {wx_cov:.1f}% non-NaN in temp_avg")
print(f"Lag vars ready   : lag1 and lag2 for temp, precip, wind")
print()
print("=" * 52)

FINAL ANALYSIS PANEL SUMMARY
Shape           : 164,400 rows x 54 columns
Sites           : 150 (well-covered: 133, low_coverage: 17)
Date range      : 2023-01-01 -> 2025-12-31

Day type breakdown:
  regular_weekday      :  545 days (49.7%)
  weekend              :  303 days (27.6%)
  school_holiday       :  212 days (19.3%)
  public_holiday       :   36 days (3.3%)

KU Leuven period breakdown (calendar days):
  teaching     : 554 days
  exam         : 182 days
  exam_prep    : 67 days
  recess       : 293 days

Weather coverage : 100.0% non-NaN in temp_avg
Lag vars ready   : lag1 and lag2 for temp, precip, wind

